# H-002 · Does a High GK / Realised Vol Ratio Predict Reversal?

Factor test for **H-002** (equities): whether a high short-window Garman–Klass (intraday OHLC) vol relative to longer-window close-to-close realised vol predicts negative next-week returns.

- **Idea** — Ratio of short-window average GK vol to longer-window realised vol; high values flag intraday stress not fully reflected in closes.
- **Claim** — High GK/realised vol ratio predicts negative next-week returns (short-horizon mean reversion).
- **Why it might work** — Wide intraday ranges with muted close-to-close vol suggest two-way fighting or intraday overreaction that partially reverses.
- **Data** — Daily OHLC long panel (`open`, `high`, `low`, `close`).

**No floor / no winsorize in the feature store:** `add_gk_vol_ratio` does **not** floor the realised-vol denominator (non-positive → NaN ratio) and does **not** winsorize the ratio. If you need winsorization, apply it in §2 (project cleaners) before or after the store call — not inside the factor API.

**`normalize` (default `True`):** when `normalize=True`, `add_gk_vol_ratio` stores a **cross-sectional (CS)** feature — the percentile rank of the mode-transformed ratio *within each date*. That is GBM-ready and comparable across names. Set `normalize=False` to keep the unranked value.

**Defaults:** `gk_window=5`, `realised_window=20` (override only with keyword args, e.g. `gk_window=10`). Realised vol is the population std of the last `realised_window` log close-to-close returns ending at `t`.

**Realised-vol std (current implementation):** population standard deviation (`ddof=0`) of the last `realised_window` log returns ending at `t`. Prefer `ddof=0` over sample std (`ddof=1`) for this factor; with `normalize=True`, CS ranks are unchanged either way. This is **not** a keyword on `add_gk_vol_ratio`.

<small>To change: edit the hard-coded `.std(ddof=0)` inside `realised_vol` in `01_data/processing/feature_implementation/gk_vol_ratio.py` (use `ddof=1` only to match a sample-std library).</small>

**`mode` options** (column `gk_vol_{mode}` — windows are **not** in the name):

| Mode | Behavior |
|------|----------|
| `ratio` (default) | Raw short-GK-mean / realised-vol (high = more intraday stress). |
| `log_ratio` | `ln` of the ratio when positive; non-positive → NaN. With `normalize=True`, CS ranks match `ratio` when the ratio is positive (monotonic). |
| `reversal` | Negated raw ratio so higher values ≈ more bullish expected return. |

Use `data.processing.feature_store.add_gk_vol_ratio` rather than reimplementing the factor inline.

## 0. Imports & Config

Resolve the repo root, configure paths / dates / factor params (`gk_window`, `realised_window`, `mode`, `normalize`), and import project helpers (`fetch_top_n_equities`, `add_gk_vol_ratio`, Alphalens).

## 1. Data Loading

Load a daily OHLC long panel for a point-in-time equity universe.

- Prefer project fetchers (`fetch_top_n_equities` / PIT S&P helpers) — no ad-hoc vendor downloads.
- Confirm columns needed by the feature store: `date`, `ticker`, `open`, `high`, `low`, `close`.

## 2. Data Cleaning & Engineering

Clean the long panel as needed (missing bars, optional winsorize, etc.) via project cleaners. Stay in **long format**.

The feature store itself does **not** winsorize or floor the denominator — do that here if desired.

Point-in-time only: features at `t` use information available at or before `t`.

## 3. Modeling / Signal Construction

Build the GK vol ratio on the cleaned panel.

### 3.1 GK / realised vol ratio (H-002)

Call `add_gk_vol_ratio` with the chosen `mode` and `normalize`.

- With `normalize=True`, output is a **CS-ranked** feature (`gk_vol_{mode}`).
- With `normalize=False`, output is the unranked mode-transformed ratio.
- Optionally run more than one `mode` (`ratio` / `log_ratio` / `reversal`) side by side for sensitivity — note which is primary.

## 4. Evaluation

Quintile spread and IC of GK vol ratio vs 5-day forward return on a fixed universe and rebalance schedule. Report net of spread/slippage (reversal is turnover-sensitive).